# Training OFA — Image Captioning Aksara Lontara
## Model: `OFA-Sys/ofa-base`

Notebook ini menjalankan **seluruh pipeline** dalam satu proses (mengikuti pola `training_git.ipynb`):
1. Mount Google Drive & Install Library (termasuk **fork transformers OFA**)
2. Pipeline: Persiapan Dataset (bbox crop) → Training → Evaluasi → Simpan ke Drive

**Konfigurasi:** Batch size 8 · Epochs 20 · LR 2e-5 · AdamW · MAX_LENGTH 128 · IMAGE_SIZE 384

> **Pastikan runtime menggunakan GPU** (Runtime → Change runtime type → GPU).
> Semua `import` berada di dalam sel Pipeline agar sel dapat dijalankan mandiri (tahan terhadap reset runtime).

> **PENTING (OFA):** OFA belum tersedia di `transformers` resmi. Sel Setup meng-install **fork** `OFA-Sys/OFA` (branch `feature/add_transformers`) yang menyediakan `OFAModel` & `OFATokenizer`. Instalasi ini akan mengganti versi `transformers` pada runtime, jadi jalankan notebook OFA di runtime terpisah dari BLIP/GIT.

---
## 1. Setup: Mount Google Drive & Install Library

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SETUP: Mount Google Drive & Install Library (+ fork transformers OFA)
# ═══════════════════════════════════════════════════════════════
import os
try:
    os.chdir("/content")
except OSError:
    pass

from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/ImageCaptioning"
LOCAL_DIR = "/content/ImageCaptioning"

# Copy dataset dari Drive ke local (lebih stabil & cepat)
assert os.path.exists(DRIVE_DIR), f"ERROR: {DRIVE_DIR} tidak ditemukan!"
!rm -rf {LOCAL_DIR}
!cp -r {DRIVE_DIR} {LOCAL_DIR}
os.chdir(LOCAL_DIR)
print(f"✅ Working directory: {os.getcwd()}")
print(f"✅ Isi folder: {os.listdir('.')}")

# Install fork transformers OFA (menyediakan OFAModel & OFATokenizer)
if not os.path.exists("/content/OFA"):
    !git clone --single-branch --branch feature/add_transformers https://github.com/OFA-Sys/OFA.git /content/OFA
!pip install -q /content/OFA/transformers/
!pip install -q Pillow nltk rouge-score tqdm

import torch
print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PIPELINE LENGKAP: Prepare → Train → Evaluate → Save (satu proses)
# Semua import berada di sel ini agar dapat dijalankan mandiri
# (tahan terhadap reset runtime Colab).
# ═══════════════════════════════════════════════════════════════
import os
import json
import random
import shutil
import warnings
from collections import defaultdict

import cv2
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib
matplotlib.rcParams["font.size"] = 11
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    OFATokenizer,
    OFAModel,
    get_linear_schedule_with_warmup,
)
from tqdm import tqdm
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score as calc_meteor
from rouge_score import rouge_scorer
import nltk

warnings.filterwarnings("ignore")
nltk.download("wordnet", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("omw-1.4", quiet=True)

# ─── Konfigurasi ───
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM: {vram:.1f} GB")

RANDOM_SEED = 42
IMAGE_SIZE = 384
MAX_LENGTH = 128
NUM_EPOCHS = 20
LR = 2e-5
BATCH_SIZE = 8
MODEL_NAME = "OFA-Sys/ofa-base"
MODEL_DIR = "model/ofa"
RAW_IMAGE_DIR = "dataset/images"
CAPTION_FILE = "dataset/captions.json"
OUTPUT_DIR = "dataset/processed"
EVAL_DIR = "hasil_evaluasi"
BBOX_PAD_RATIO = 0.20
PROMPT = " what does the image describe?"

# OFA memakai normalisasi torchvision (mean/std 0.5), BUKAN image processor HF.
OFA_MEAN = [0.5, 0.5, 0.5]
OFA_STD = [0.5, 0.5, 0.5]
patch_resize_transform = transforms.Compose([
    lambda image: image.convert("RGB"),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=OFA_MEAN, std=OFA_STD),
])

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(RANDOM_SEED)

os.makedirs(EVAL_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)


# ═══════════════════════════════════════════════════════════════
# UTIL: Deteksi bounding box karakter + crop ke canvas 384x384
# (HARUS sama dgn `inference_ofa.ipynb` agar distribusi training
#  konsisten dengan distribusi inference per-karakter.)
# ═══════════════════════════════════════════════════════════════
def _binary_mask(pil_image):
    img_np = np.array(pil_image.convert("RGB"))
    img_cv = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
    gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    thresh = cv2.adaptiveThreshold(
        blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, blockSize=25, C=10
    )
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=1)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=1)
    return thresh


def detect_bbox(pil_image, pad_ratio=BBOX_PAD_RATIO):
    """Deteksi bounding box ketat dari karakter pada gambar PIL.

    Returns (x1, y1, x2, y2) dengan padding pad_ratio dari sisi bbox.
    Fallback ke full image jika tidak ada piksel aktif.
    """
    mask = _binary_mask(pil_image)
    h, w = mask.shape
    cols = np.where(np.any(mask > 0, axis=0))[0]
    rows = np.where(np.any(mask > 0, axis=1))[0]
    if len(cols) == 0 or len(rows) == 0:
        return (0, 0, w, h)
    x1, x2 = int(cols[0]), int(cols[-1]) + 1
    y1, y2 = int(rows[0]), int(rows[-1]) + 1
    bw, bh = x2 - x1, y2 - y1
    pad_x = max(5, int(bw * pad_ratio))
    pad_y = max(5, int(bh * pad_ratio))
    x1 = max(0, x1 - pad_x)
    y1 = max(0, y1 - pad_y)
    x2 = min(w, x2 + pad_x)
    y2 = min(h, y2 + pad_y)
    return (x1, y1, x2, y2)


def crop_to_canvas(pil_image, canvas_size=IMAGE_SIZE, pad_ratio=BBOX_PAD_RATIO):
    """Crop ke bbox karakter lalu stretch ke canvas_size x canvas_size.

    Identik dengan preprocessing yang dipakai `inference_ofa.ipynb` pada
    setiap karakter hasil segmentasi multi-character.
    """
    x1, y1, x2, y2 = detect_bbox(pil_image, pad_ratio=pad_ratio)
    cropped = pil_image.crop((x1, y1, x2, y2))
    return cropped.resize((canvas_size, canvas_size), Image.LANCZOS)


# ═══════════════════════════════════════════════════════════════
# TAHAP 1: PERSIAPAN DATASET (dengan bounding-box crop)
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("  TAHAP 1: PERSIAPAN DATASET")
print("=" * 60)

with open(CAPTION_FILE, "r", encoding="utf-8") as f:
    raw_data = json.load(f)
print(f"Entri caption: {len(raw_data)}")

flat_data = []
stat_single = 0
stat_multi = 0
for item in raw_data:
    img_id = item["images"]
    img_filename = f"{img_id}.png"
    karakter_list = item.get("karakter_list", [])
    jumlah = item.get("jumlah_karakter", len(karakter_list))
    if jumlah <= 1:
        stat_single += 1
    else:
        stat_multi += 1
    for caption in item["captions"]:
        flat_data.append({
            "image": img_filename,
            "caption": caption,
            "karakter_list": karakter_list,
            "jumlah_karakter": jumlah
        })

print(f"Total pasang (image, caption): {len(flat_data)}")
print(f"Gambar 1 karakter: {stat_single}")
print(f"Gambar 2 karakter: {stat_multi}")

seen_images = {}
for item in flat_data:
    img = item["image"]
    if img not in seen_images:
        seen_images[img] = item["jumlah_karakter"]

img_by_type = defaultdict(list)
for img, jml in seen_images.items():
    img_by_type[jml].append(img)

train_imgs, val_imgs, test_imgs = set(), set(), set()
for jml, imgs in img_by_type.items():
    random.shuffle(imgs)
    n = len(imgs)
    n_train = max(1, int(n * 0.8))
    n_val = max(1, int(n * 0.1))
    train_imgs.update(imgs[:n_train])
    val_imgs.update(imgs[n_train:n_train + n_val])
    test_imgs.update(imgs[n_train + n_val:])

train_data = [d for d in flat_data if d["image"] in train_imgs]
val_data = [d for d in flat_data if d["image"] in val_imgs]
test_data = [d for d in flat_data if d["image"] in test_imgs]

print(f"\nSplit: Train={len(train_data)}, Val={len(val_data)}, Test={len(test_data)}")

for name, data in [("train", train_data), ("val", val_data), ("test", test_data)]:
    split_dir = f"{OUTPUT_DIR}/{name}"
    os.makedirs(f"{split_dir}/images", exist_ok=True)
    with open(f"{split_dir}/captions_{name}.json", "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

# Crop bounding box -> resize 384x384 untuk setiap split
for name, data in [("train", train_data), ("val", val_data), ("test", test_data)]:
    processed = set()
    errors = []
    for item in data:
        img = item["image"]
        if img in processed:
            continue
        processed.add(img)
        src = os.path.join(RAW_IMAGE_DIR, img)
        dst = os.path.join(OUTPUT_DIR, name, "images", img)
        if os.path.exists(src):
            try:
                im = Image.open(src).convert("RGB")
                im = crop_to_canvas(im, canvas_size=IMAGE_SIZE)
                im.save(dst, format="PNG")
                Image.open(dst).verify()
            except Exception as e:
                errors.append(f"{img}: {e}")
        else:
            errors.append(f"{img}: file tidak ditemukan")
    print(f"  {name}: {len(processed)} gambar" + (f" ({len(errors)} error)" if errors else ""))

print("✅ Persiapan dataset selesai!")

# ─── Verifikasi visual bbox crop (6 sampel training) ───
sample = sorted(list(train_imgs))[:6]
if sample:
    fig, axes = plt.subplots(2, len(sample), figsize=(3 * len(sample), 6))
    if len(sample) == 1:
        axes = np.array(axes).reshape(2, 1)
    for i, img_name in enumerate(sample):
        orig = Image.open(os.path.join(RAW_IMAGE_DIR, img_name)).convert("RGB")
        x1, y1, x2, y2 = detect_bbox(orig)
        orig_draw = orig.copy()
        ImageDraw.Draw(orig_draw).rectangle([x1, y1, x2, y2], outline="red", width=3)
        cropped = Image.open(os.path.join(OUTPUT_DIR, "train", "images", img_name))
        axes[0, i].imshow(orig_draw)
        axes[0, i].set_title(f"{img_name}\n(orig + bbox)", fontsize=8)
        axes[0, i].axis("off")
        axes[1, i].imshow(cropped)
        axes[1, i].set_title("crop → 384×384", fontsize=8)
        axes[1, i].axis("off")
    plt.suptitle("Verifikasi Bounding Box Crop (Training)", fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{EVAL_DIR}/bbox_verify_ofa.png", dpi=120, bbox_inches="tight")
    plt.show()


# ═══════════════════════════════════════════════════════════════
# TAHAP 2: FINE-TUNING OFA
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("  TAHAP 2: FINE-TUNING OFA")
print("=" * 60)


class OFACaptionDataset(Dataset):
    """Dataset untuk OFA (encoder-decoder: prompt tetap + patch_images).

    OFA tidak menghitung loss secara otomatis & tidak menerima `labels`,
    sehingga di sini disiapkan:
      - patch_image       : tensor gambar ter-normalisasi (3,384,384)
      - prompt_input_ids  : token instruksi tetap (encoder input)
      - decoder_input_ids : target di-geser-kanan (diawali BOS)
      - labels            : target asli (pad → -100)
    """
    def __init__(self, json_file, image_dir, tokenizer, max_length=128):
        with open(json_file, "r", encoding="utf-8") as f:
            self.data = json.load(f)
        self.image_dir = image_dir
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.prompt_ids = tokenizer([PROMPT], return_tensors="pt").input_ids[0]
        print(f"  Dataset: {len(self.data)} pasang")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        img_path = os.path.join(self.image_dir, os.path.basename(item["image"]))
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            image = Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE), (255, 255, 255))

        patch_image = patch_resize_transform(image)
        # OFATokenizer (BartTokenizer) menambahkan BOS(0) di awal & EOS(2) di akhir.
        full = self.tokenizer(
            item["caption"], padding="max_length", truncation=True,
            max_length=self.max_length, return_tensors="pt"
        ).input_ids[0]
        decoder_input_ids = full[:-1].clone()
        labels = full[1:].clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        return {
            "patch_image": patch_image,
            "prompt_input_ids": self.prompt_ids.clone(),
            "decoder_input_ids": decoder_input_ids,
            "labels": labels,
        }


def ofa_generate(gen_model, gen_tokenizer, image, max_length=MAX_LENGTH, num_beams=4):
    """Hasilkan caption dari satu gambar PIL menggunakan OFA."""
    patch_image = patch_resize_transform(image).unsqueeze(0).to(DEVICE)
    prompt_ids = gen_tokenizer([PROMPT], return_tensors="pt").input_ids.to(DEVICE)
    with torch.no_grad():
        gen = gen_model.generate(
            prompt_ids,
            patch_images=patch_image,
            max_length=max_length,
            min_length=5,
            num_beams=num_beams,
            early_stopping=True,
            no_repeat_ngram_size=2,
        )
    return gen_tokenizer.batch_decode(gen, skip_special_tokens=True)[0].strip()


# Load model
print("Memuat OFA...")
tokenizer = OFATokenizer.from_pretrained(MODEL_NAME)
model = OFAModel.from_pretrained(MODEL_NAME, use_cache=False).to(DEVICE)
print(f"Parameter: {sum(p.numel() for p in model.parameters()):,}")

# Simpan tokenizer
tokenizer.save_pretrained(f"{MODEL_DIR}/best_model")

# DataLoader
train_ds = OFACaptionDataset(f"{OUTPUT_DIR}/train/captions_train.json", f"{OUTPUT_DIR}/train/images", tokenizer, MAX_LENGTH)
val_ds = OFACaptionDataset(f"{OUTPUT_DIR}/val/captions_val.json", f"{OUTPUT_DIR}/val/images", tokenizer, MAX_LENGTH)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

# Training
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=100, num_training_steps=total_steps)

history = {"train_loss": [], "val_loss": []}
best_val_loss = float("inf")

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} Train", leave=False):
        patch_images = batch["patch_image"].to(DEVICE)
        prompt_ids = batch["prompt_input_ids"].to(DEVICE)
        dec_in = batch["decoder_input_ids"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        patch_masks = torch.ones(patch_images.size(0), dtype=torch.bool, device=DEVICE)
        outputs = model(
            input_ids=prompt_ids,
            patch_images=patch_images,
            patch_masks=patch_masks,
            decoder_input_ids=dec_in,
        )
        logits = outputs.logits
        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)), labels.reshape(-1), ignore_index=-100
        )
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    train_loss = total_loss / len(train_loader)

    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} Val", leave=False):
            patch_images = batch["patch_image"].to(DEVICE)
            prompt_ids = batch["prompt_input_ids"].to(DEVICE)
            dec_in = batch["decoder_input_ids"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            patch_masks = torch.ones(patch_images.size(0), dtype=torch.bool, device=DEVICE)
            outputs = model(
                input_ids=prompt_ids,
                patch_images=patch_images,
                patch_masks=patch_masks,
                decoder_input_ids=dec_in,
            )
            logits = outputs.logits
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)), labels.reshape(-1), ignore_index=-100
            )
            total_loss += loss.item()
    val_loss = total_loss / len(val_loader)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        os.makedirs(f"{MODEL_DIR}/best_model", exist_ok=True)
        model.save_pretrained(f"{MODEL_DIR}/best_model")
        tokenizer.save_pretrained(f"{MODEL_DIR}/best_model")
        print(f"  -> Best model! Val Loss: {best_val_loss:.4f}")

    if epoch % 5 == 0:
        ckpt = f"{MODEL_DIR}/checkpoint_epoch_{epoch}"
        os.makedirs(ckpt, exist_ok=True)
        model.save_pretrained(ckpt)

with open(f"{MODEL_DIR}/training_history.json", "w") as f:
    json.dump(history, f, indent=2)

print(f"\nTraining selesai! Best Val Loss: {best_val_loss:.4f}")


# ═══════════════════════════════════════════════════════════════
# TAHAP 3: EVALUASI PADA TEST SET (load best_model dari disk)
# ═══════════════════════════════════════════════════════════════
def compute_metrics(predictions, test_gambar, filter_jml=None):
    """Hitung BLEU-1..4, METEOR, ROUGE-L."""
    refs_all, hyps_all = [], []
    for img_key, pred in predictions.items():
        info = test_gambar[img_key]
        if filter_jml is not None and info["jumlah_karakter"] != filter_jml:
            continue
        refs = [c.lower().split() for c in info["captions"]]
        hyp = pred.lower().split()
        refs_all.append(refs)
        hyps_all.append(hyp)

    if not refs_all:
        return {k: 0 for k in ["BLEU-1","BLEU-2","BLEU-3","BLEU-4","METEOR","ROUGE-L"]}

    sm = SmoothingFunction().method1
    b1 = corpus_bleu(refs_all, hyps_all, weights=(1,0,0,0), smoothing_function=sm) * 100
    b2 = corpus_bleu(refs_all, hyps_all, weights=(0.5,0.5,0,0), smoothing_function=sm) * 100
    b3 = corpus_bleu(refs_all, hyps_all, weights=(0.33,0.33,0.33,0), smoothing_function=sm) * 100
    b4 = corpus_bleu(refs_all, hyps_all, weights=(0.25,0.25,0.25,0.25), smoothing_function=sm) * 100
    met = np.mean([calc_meteor(r, h) for r, h in zip(refs_all, hyps_all)]) * 100
    rsc = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
    rg = []
    for refs, hyp in zip(refs_all, hyps_all):
        hyp_s = " ".join(hyp)
        rg.append(max(rsc.score(" ".join(r), hyp_s)["rougeL"].fmeasure for r in refs))
    rouge = np.mean(rg) * 100
    return {"BLEU-1": b1, "BLEU-2": b2, "BLEU-3": b3, "BLEU-4": b4, "METEOR": met, "ROUGE-L": rouge}


print("\n" + "=" * 60)
print("  TAHAP 3: EVALUASI OFA (best model dari disk)")
print("=" * 60)

# Bebaskan memori model in-memory dulu (final-epoch) → load best dari disk
del model
if DEVICE == "cuda":
    torch.cuda.empty_cache()

best_dir = f"{MODEL_DIR}/best_model"
eval_tokenizer = OFATokenizer.from_pretrained(best_dir)
eval_model = OFAModel.from_pretrained(best_dir, use_cache=False).to(DEVICE)
eval_model.eval()
print(f"Best model dimuat dari: {best_dir}")

# Load test data
with open(f"{OUTPUT_DIR}/test/captions_test.json", "r", encoding="utf-8") as f:
    test_items = json.load(f)

test_gambar = {}
for item in test_items:
    key = item["image"]
    if key not in test_gambar:
        test_gambar[key] = {"captions": [], "jumlah_karakter": item.get("jumlah_karakter", 1)}
    test_gambar[key]["captions"].append(item["caption"])

# Generate captions
predictions = {}
for img_key in tqdm(test_gambar, desc="Evaluasi"):
    img_path = os.path.join(f"{OUTPUT_DIR}/test/images", os.path.basename(img_key))
    try:
        image = Image.open(img_path).convert("RGB")
    except Exception:
        continue
    predictions[img_key] = ofa_generate(eval_model, eval_tokenizer, image)

# Sanity check: print 3 contoh prediksi
print("\n--- Sample Prediksi (sanity check) ---")
for i, (k, v) in enumerate(predictions.items()):
    if i >= 3:
        break
    print(f"  [{k}] -> {v!r}")

# Hitung metrik (kategori 2_karakter dihilangkan: dataset hanya single-char)
results = {
    "keseluruhan": compute_metrics(predictions, test_gambar),
    "1_karakter": compute_metrics(predictions, test_gambar, filter_jml=1),
}

print("\n--- Hasil Evaluasi OFA ---")
for kategori, metrik in results.items():
    print(f"\n  [{kategori}]")
    for k, v in metrik.items():
        print(f"    {k}: {v:.2f}%")

# Simpan
with open(f"{EVAL_DIR}/evaluasi_ofa.json", "w", encoding="utf-8") as f:
    json.dump({"metrik": results, "sample_predictions": {k: predictions[k] for k in list(predictions)[:5]}},
              f, ensure_ascii=False, indent=2)

# Plot training curve
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
epochs_range = range(1, len(history["train_loss"]) + 1)
ax.plot(epochs_range, history["train_loss"], "o-", label="Train Loss", color="#4CAF50")
ax.plot(epochs_range, history["val_loss"], "s-", label="Val Loss", color="#FF5722")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("OFA - Training & Validation Loss", fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{EVAL_DIR}/training_curve_ofa.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nHasil disimpan: {EVAL_DIR}/evaluasi_ofa.json")


# ═══════════════════════════════════════════════════════════════
# TAHAP 4: SIMPAN KE GOOGLE DRIVE
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("  TAHAP 4: SIMPAN KE GOOGLE DRIVE")
print("=" * 60)

DRIVE_DIR = "/content/drive/MyDrive/ImageCaptioning"
LOCAL_DIR = "/content/ImageCaptioning"

for folder in [MODEL_DIR, EVAL_DIR]:
    src = os.path.join(LOCAL_DIR, folder)
    dst = os.path.join(DRIVE_DIR, folder)
    if os.path.exists(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"  {folder}/ -> Drive")

print("\n✅ SELURUH PIPELINE SELESAI! Semua hasil tersimpan di Google Drive.")